In [ ]:
import pymupdf
import re
from dotenv import load_dotenv

load_dotenv()


def is_toc_page_fast(page):
    #TODO: Check this logic with more books
    text = page.get_text("text")

    if not text:
        return False

    text_lower = text.lower()

    if "table of contents" in text_lower or text_lower.strip().startswith("contents"):
        return True

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    pattern = re.compile(r".+\.{2,}\s*\d+$|.+\s+\d+$")
    matches = sum(1 for line in lines if pattern.match(line))

    if len(lines) > 3 and matches / len(lines) > 0.2:
        return True

    return False


def find_toc_pages_pymupdf(pdf_path):
    toc_pages = []

    doc = pymupdf.open(pdf_path)

    for i in range(len(doc)):
        page = doc[i]

        if is_toc_page_fast(page):
            toc_pages.append(i)
        elif toc_pages:
            break  #The assumption is that the table of contents is contiguous
    doc.close()

    return toc_pages

In [2]:
toc_pages = find_toc_pages_pymupdf("example1.pdf")

print(toc_pages)

[5, 6, 7, 8, 9]


In [3]:
system_prompt_toc_parser = """
You are a meticulous document boundary analyst. You will receive:
- A list of texts. Each item in the list represents a page of the table of contents in a book.

Your task:
- Read through the contents of the page. 
- Determine the most likely start page where the subchapter begins (usually at the first occurrence of its heading/title or its numbering)
- Determine the most likely end page where the subchapter ends. The end page is typically the page immediately before the next subchapter's start page. If the subchapter is not present in the window, end at the last page where the subchapter's content is predominant.
- Use textual evidence and be resilient to OCR errors.
- If evidence is ambiguous, use logic and the page number of adjacent chapters to provide best effort estimates

Output requirements (CRITICAL):
- Output the data into valid JSON (no markdown, no extra keys beyond provided schema).
- Include the confidence as a number from 0 to 1
- Never invent pages outside the provided page range. Make sure that page numbers at most overlap by only 1 page (to account for a chapter ending and a new chapter starting in the same page)
- Never have a starting page larger than an ending page. This is an impossible case

JSON schema:
[
    {
        'part_name': None,
        'chapter_name': 'Chapter 2. Managing Users and Permissions',
        'subchapter': 'Understanding the purpose of users and groups',
        'page_start': '32',
        'page_end': '32',
        'confidence': 0.923,
    },
    {
        'part_name': None,
        'chapter_name': 'Chapter 3. Navigating and Essential Commands',
        'subchapter': 'Learning the essential Linux commands',
        'page_start': '85',
        'page_end': '89',
        'confidence': 0.988,
    },
    {
        'part_name': 'Part One: The Dimensions of Reading',
        'chapter_name': '1. The Activite and Art of Reading',
        'subchapter': None,
        'page_start': 3,
        'page_end': 15,
        'confidence': 0.765,
    }
]
Now produce the JSON for all provided TOC subchapter items, preserving the given order
"""

system_prompt_toc_parser_corrector = """
You are a strict JSON post-processor and structural validator. You will receive:
- A JSON array generated from a table of contents parser.

Your task:
- Validate and fix structural inconsistencies in the JSON.
- Ensure logical hierarchy between parts, chapters, and subchapters.

Validations to check (CRITICAL):
- Parts must have:
    - part_name != None
    - chapter_name == None
    - subchapter == None
- Chapters must have:
    - chapter_name != None
    - subchapter == None
    - part_name may or may not be present depending on grouping and whether the original table of contents had parts to begin wtih.
- Subchapters must have:
    - subchapter ≠ None
    - chapter_name ≠ None

Changes to consider:
- If an entry has both part_name and chapter_name, treat it as a chapter (set part_name to None).
- If an entry has chapter_name but also a non-null subchapter, keep as subchapter.
- If an entry has only part_name, enforce chapter_name=None and subchapter=None.
- If unclear, infer the most likely type using naming patterns (e.g., "Part", "Chapter", numbering).

Changes to avoid:
- Do not modify page_start or page_end unless: page_start > page_end → swap them. Preserve original page ranges otherwise.

Output requirements (CRITICAL):
- Output the data into valid JSON (no markdown, no extra keys beyond provided schema).
- Include the confidence as a number from 0 to 1
- Never invent pages outside the provided page range. Make sure that page numbers at most overlap by only 1 page (to account for a chapter ending and a new chapter starting in the same page)
- Never have a starting page larger than an ending page. This is an impossible case

JSON schema:
[
    {
        'part_name': None,
        'chapter_name': 'Chapter 2. Managing Users and Permissions',
        'subchapter': 'Understanding the purpose of users and groups',
        'page_start': '32',
        'page_end': '32',
        'confidence': 0.923,
    },
    {
        'part_name': None,
        'chapter_name': 'Chapter 3. Navigating and Essential Commands',
        'subchapter': 'Learning the essential Linux commands',
        'page_start': '85',
        'page_end': '89',
        'confidence': 0.988,
    },
    {
        'part_name': 'Part One: The Dimensions of Reading',
        'chapter_name': '1. The Activite and Art of Reading',
        'subchapter': None,
        'page_start': 3,
        'page_end': 15,
        'confidence': 0.765,
    }
]
"""

system_prompt_concepts_summarizer = """
You are an expert technical educator and summarizer. You will receive:
- A subchapter title (and optional numbering)
- The extracted text for that subchapter, broken into pages (page number + text)
- Possibly noisy OCR; preserve the meaning, not the exact wording

Your job:
- Produce a concise but faithful summary of the concepts introduced in the subchapter.
- Do NOT summarize everything verbatim; instead extract the key concepts, definitions, and relationships.
- Focus on “pre-read” value: what a learner needs to know before deeper reading.
- Use the page text as the source of truth. If something is unclear/contradicted, mention that uncertainty briefly.

Output requirements (CRITICAL):
- Output ONLY valid JSON (no markdown).
- Include a single concise paragraph that summarizes the key concepts introduced in the subchapter and explains their relationships at a learner-friendly level. Prioritize definitions, major claims, and the “so what / how it connects” ideas. Do not quote large passages verbatim; instead paraphrase. If the text is incomplete or ambiguous due to OCR, still provide the best-faith summary, but clearly and briefly note uncertainty in a natural way.

JSON Schema:
{
    'summary': 'We learn about simple Linux filesystem commands to help us become more confident in navigating the filesystem of our Ubuntu server. In this subchapter, the commands `cd`, `pwd`, `touch`, `rm`, `ls` and `ls -l` are introduced. 
}
"""

system_prompt_question_generator = """
You are a question-writing tutor for self-study. You will receive:
- A subchapter title
- Extracted page text, provided as a list of pages (each with a page number and text)
- A summary of the subchapter generated by a previous LLM

Your task:
- Generate approximately 5 questions per page that help the user pre-read and test understanding.
- Questions must be answerable from the provided page text (or based on explicit explanations present there).
- Mix question types to improve learning:
    - factual recall
    - conceptual explanation (“why/how”)
    - application/mini-scenarios
    - comparison/contrast (when the page supports it)
    - “common misconception” style (only if the text indicates it)
- Do NOT provide answers. Do NOT provide page-level summaries as answers.
- Keep questions clear even if OCR is imperfect; infer only when strongly supported.

Output requirements (CRITICAL):
- Output ONLY valid JSON (no markdown).
- For each question, include:
    - question (string)
    - type (one of: "recall" | "concept" | "application" | "comparison" | "misconception")
    - difficulty (1–3)
Use this schema:
{
    "title": "string",
    "questions": [
        {
            "question": "What command would you use to create a file in Linux?",
            "type": "recall, application",
            "difficulty": 1,
        },
        {
            "question": "What is the difference between `ls` and `ls -a`",
            "type": "comparison",
            "difficulty": 2,
        },
        {
            "question": "How would you update the 'last updated' time of a file without making any changes to it?",
            "type": "application",
            "difficulty": 3,
        }
    ]
}
"""

In [4]:
toc_pages

[5, 6, 7, 8, 9]

In [5]:
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

class ChapterPage(BaseModel):
    part_name: Optional[str] = Field(
        default=None,
        description="Part of the book. Consists of chapters. Not present in all Table of Contents"
    )
    chapter_name: Optional[str] = Field(
        default=None,
        description="Full chapter title including numbering (if present). Consists of subchapters."
    )
    subchapter: Optional[str] = Field(
        default=None,
        description="Subchapter or section title (if present)"
    )
    page_start: int = Field(
        description="Starting page number of the section"
    )
    page_end: int = Field(
        description="Ending page number of the section"
    )
    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence score between 0 and 1"
    )

class ChapterPages(BaseModel):
    chapter_page: List[ChapterPage] = Field(
        default=[],
        description="List of ChapterPage items"
    )

class ChapterSummary(BaseModel):
    summary: str = Field(
        description="Summary of the chapter, condensed into one paragraph."
    )

class ChapterQuestion(BaseModel):
    question: str = Field(description="The question text")
    type: List[Literal["recall", "concept", "application", "comparison", "misconception"]] = Field(
        description="Type of question"
    )
    difficulty: int = Field(
        ge=1,
        le=5,
        description="Difficulty level from 1 (easy) to 5 (extremely hard)"
    )

In [51]:
toc_pages
doc = pymupdf.open('example1.pdf')

In [52]:
toc_page_content = ''.join(doc[i].get_text() for i in toc_pages)

In [ ]:
from openai import OpenAI

model = OpenAI()

response = model.beta.chat.completions.parse(
    model='gpt-5.4-nano',
    messages=[
        {'role':'system', 'content': system_prompt_toc_parser},
        {'role':'user', 'content': toc_page_content}
    ],
    response_format=ChapterPages
)

response = model.beta.chat.completions.parse(
    model='gpt-5.4-nano',
    messages=[
        {'role':'system', 'content': system_prompt_toc_parser_corrector},
        {'role':'user', 'content': response.choices[0].message.content}
    ],
    response_format=ChapterPages
)


In [ ]:
print(response.choices[0].message.parsed)


chapter_page=[ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='Why is Postgres so popular?', page_start=4, page_end=5, confidence=0.72), ChapterPage(part_name=None, chapter_name='1 Meeting Postgres', subchapter='“Just use Postgres!” explained', page_start=5, page_end=5, confidence=0.2), ChapterPage(part_name=None, chapter_name='1 Meeting Postgres', subchapter='Starting Postgres in Docker', page_start=6, page_end=9, confidence=0.58), ChapterPage(part_name=None, chapter_name='1 Meeting Postgres', subchapter='Connecting with psql', page_start=10, page_end=10, confidence=0.75), ChapterPage(part_name=None, chapter_name='1 Meeting Postgres', subchapter='Generating mock data', page_start=11, page_end=14, confidence=0.85), ChapterPage(part_name=None, chapter_name='1 Meeting Postgres', subchapter='Running basic queries', page_start=15, page_end=17, confidence=0.8), ChapterPage(part_name='Part 1 Postgres as a relational database', c

In [58]:
for chapter_page in response.choices[0].message.parsed.chapter_page[5:]:
    print("================")
    print(chapter_page.subchapter)
    print("================")
    for page in range(chapter_page.page_start, chapter_page.page_end+1):
        print(f"Page {page}")
        print(f"{doc[page].get_text()[:200]}")

Running basic queries
Page 15
xiv
acknowledgments
xiv
author and university educator, who worked as technical editors on the book. Finally, 
I am thankful to the many independent experts and professionals who volunteered to 
read 
Page 16
xv
about this book
In 2009, I graduated from university and landed my first full-time job as a Java devel­
oper. Our team chose Postgres for a high-load web application I had yet to build. In 
those d
Page 17
xvi
about this book
xvi
¡ The book doesn’t cover all existing Postgres capabilities. Instead, it walks you 
through some of the most prominent features that are widely used. And the book 
does this at
Creating the database structure
Page 19
xviii
about this book
xviii
About the code
This book contains many examples of source code both in numbered listings and in 
line with normal text. In both cases, source code is formatted in a fixed-w
Page 20
xix
about the author
Denis Magda is a software engineer who started his 
career at Sun Microsystems an

In [ ]:
def validate_toc(chapter_pages: ChapterPages) -> ChapterPages:
    fixed_chapter_pages = []
    prev_end_page = None
    current_part = None
    current_chapter = None

    for chapter_page in chapter_pages:
        part = chapter_page.part_name
        chapter = chapter_page.chapter_name
        subchapter = chapter_page.subchapter

        #If part is not None, consider that as latest part
        if part is not None:
            current_part = part

        #If part is None, add latest part in
        if part is None and chapter is not None:
            current_chapter = chapter
            chapter_page.part_name = current_part
        elif part is None and chapter is None and subchapter is not None:
            chapter_page.part_name = current_part
            chapter_page.chapter_name = current_chapter
        
        #(Rare case) Account for start and end pages being flipped. Identified by start page being larger than end page
        start_page = chapter_page.page_start
        end_page = chapter_page.page_end

        if start_page > end_page:
            chapter_page.page_start, chapter_page.page_end = end_page, start_page
            start_page, end_page = chapter_page.page_start, chapter_page.page_end

        #Continuity check (consecutive ChapterPage items should have at max 1 page overlap, which is the case where old chapter end and new chapter start at the same page.)
        if prev_end_page is not None:
            if start_page < prev_end_page - 1:
                fixed_chapter_pages.append(chapter_page)
                return False, fixed_chapter_pages

        prev_end_page = end_page
        fixed_chapter_pages.append(chapter_page)
    return True, fixed_chapter_pages

In [ ]:
is_success, fixed_chapter_pages = validate_toc(response.choices[0].message.parsed.chapter_page)
if not is_success:
    raise ValueError("Something is wrong, pages are not continuous")

In [83]:
fixed_chapter_pages

[ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='Why is Postgres so popular?', page_start=4, page_end=5, confidence=0.72),
 ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='“Just use Postgres!” explained', page_start=5, page_end=5, confidence=0.2),
 ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='Starting Postgres in Docker', page_start=6, page_end=9, confidence=0.58),
 ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='Connecting with psql', page_start=10, page_end=10, confidence=0.75),
 ChapterPage(part_name='Part 1 Postgres as a relational database', chapter_name='1 Meeting Postgres', subchapter='Generating mock data', page_start=11, page_end=14, confidence=0.85),
 ChapterPage(part_name='Part 1 Postgres as a relational database', c

In [ ]:
def generate_summary(chapter_pages: ChapterPages) -> ChapterPages:
    for chapter_page in chapter_pages:
        text_to_summarize = ''
        pages = doc.pages(chapter_page.page_start+20, chapter_page.page_end+21)
        for page in pages:
            text_to_summarize += page.get_text()
            text_to_summarize += '\n\n'

        print(f"{chapter_page.page_start}, {chapter_page.page_end} ---- {text_to_summarize}")



In [87]:
generate_summary(fixed_chapter_pages)

4, 5 ---- 3
1
Meeting Postgres
This chapter covers
¡ Understanding what “just use Postgres” means
¡ Starting Postgres and connecting from a 
command-line tool
¡ Generating mock data using built-in database 
capabilities and running a few basic queries
PostgreSQL (or simply Postgres, as most people call it) is one of the fastest-growing 
relational and general-purpose databases. As a relational database, it natively supports 
Structured Query Language (SQL) and is a popular choice for online transaction 
processing (OLTP) workloads that process user requests with low latency while 
guaranteeing data consistency and integrity even if a database server crashes in the 
middle of an operation execution. You encounter OLTP systems daily when you, for 
example, book a flight to your next vacation destination, pay for groceries at a local 
supermarket, or share a new post on social media.
Over the years, Postgres has evolved into a general-purpose database that supports 
use cases beyond tradi